# 00 · Quickstart (offline, simulated tenant)

Runs the whole evaluation pipeline against a **simulated tenant**: Copilot Chat answers, the DAX Fabric IQ runs,
workspace monitoring with ingestion delay, the judge model and Foundry Evaluations. No sign-in and no Azure
resources, so you can show the flow anywhere and see exactly what each option records.

It runs the Contoso golden set (8 questions over three semantic models plus decoys) and scores every answer with
**both** options:

* **Option 1 · Fabric-native**: deterministic Python scorers plus a Foundry-hosted judge model (simulated here).
* **Option 2 · Foundry Evaluations**: the same scorers registered as Foundry code evaluators, plus prompt and
  built-in evaluators (run locally by an emulator here).

In [ ]:
# Make the cfeval package available.
# - Local clone (VS Code / Jupyter): the cell below finds src/ in the repo.
# - Fabric notebook: uncomment the %pip line (or attach an Environment with the package).
# %pip install -q "cfeval @ git+https://github.com/bgbailey/copilot_iq_eval"
import sys, pathlib
repo = pathlib.Path.cwd()
while repo != repo.parent and not (repo / "src" / "cfeval").exists():
    repo = repo.parent
if (repo / "src" / "cfeval").exists() and str(repo / "src") not in sys.path:
    sys.path.insert(0, str(repo / "src"))
import cfeval
print("cfeval", cfeval.__version__)

## Run the golden set (baseline behaviour)
`profile` picks the simulated behaviour: `baseline`, `improving` or `tuned`.

In [ ]:
from cfeval.cli import build_harness
from cfeval.config import Settings

settings = Settings(output_dir="runs")  # Offline defaults, never ambient live settings.
outcome = build_harness(settings, option="both", mock=True, publish=False, profile="baseline").run()
outcome.metrics

## Results
One row per ask: the verdict and why, what Copilot cited, which semantic models Fabric IQ queried, latency and DAX.

In [ ]:
from pprint import pprint

cols = ["QuestionId", "Verdict", "ValueScore", "ArtifactScore", "ModelScore", "CitedArtifacts", "QueriedModels",
        "LatencyMs", "DaxQueryCount", "DaxTotalMs", "Reasons"]
pprint([{key: row[key] for key in cols} for row in outcome.results])

## One ask end to end
The question, Copilot's answer and citations, its own explanation, and the DAX Fabric IQ ran for it.

In [ ]:
from cfeval.sinks import JsonlSink

run = JsonlSink(outcome.folder)
ask = next(a for a in run.read("EvalAsk") if a["QuestionId"] == "Q03")
print("Question:", ask["Question"])
print("Answer:  ", ask["Answer"])
print("Cited:   ", ask["ArtifactNames"] or "(nothing)")
print("Why:     ", ask["Why"])
for q in run.read("FabricIqQuery"):
    if q["AskId"] == ask["AskId"]:
        print("\nDAX on %s (%.0f ms, ended %.1f s after the ask started):\n%s" % (
            q["ItemName"], q["DurationMs"], q["OffsetMs"] / 1000, q["Dax"]))

## The run summary written to the run folder

In [ ]:
print(outcome.summary_text())

## Next
* `01_setup_eventhouse` creates the telemetry tables, OneLake availability and the lakehouse shortcuts the app reads.
* `02_option1_fabric_native` and `03_option2_foundry` run the same pipeline against Copilot Chat.
* The default live acceptance route is `tools/setup_demo.py` then one `tools/rehearse_demo.py` run (see docs/DEMO.md).
* Simulated history is labelled; an aggregate improvement curve is not a measured live result.